# Interactive Agent Session: Chapter 2 — LangGraph Intelligent Support Router

**The Business Problem:** Customer support tickets get misrouted 40% of the time, costing $2M/year in delays. High-severity VIP tickets sit in the same queue as password resets.

**The Solution:** A LangGraph state machine with 6 nodes: Sentiment Analysis → Intent Classification → Priority Scoring → Specialized Handlers (Refund / Technical / Billing) → VIP Escalation.

**Key Concept:** Unlike LLM chains, LangGraph gives you **deterministic control flow** — you decide exactly which path a ticket takes based on conditional edges.

In [ ]:
import sys
!{sys.executable} -m pip install --quiet langchain langchain_openai langgraph python-dotenv

In [ ]:
%%capture lg_logs

import os
from typing import TypedDict, List, Literal
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

load_dotenv()
USE_SIMULATION = not os.getenv("OPENAI_API_KEY")
if not USE_SIMULATION:
    llm = ChatOpenAI(model="gpt-4o-mini", max_tokens=150)

# ── ADVANCED STATE SCHEMA ──
class SupportState(TypedDict):
    ticket_text: str
    customer_tier: str  # "standard" or "vip"
    sentiment: str  # "angry", "frustrated", "neutral"
    intent: str  # "refund", "technical", "billing", "account"
    priority: str  # "P0_CRITICAL", "P1_HIGH", "P2_MEDIUM", "P3_LOW"
    history: List[str]
    response: str
    escalated: bool

# ── NODE 1: Sentiment Detector ──
def sentiment_node(state: SupportState):
    text = state["ticket_text"].lower()
    if USE_SIMULATION:
        if any(w in text for w in ["furious", "unacceptable", "lawsuit", "worst", "scam"]):
            sentiment = "angry"
        elif any(w in text for w in ["frustrated", "annoyed", "waiting", "still", "again"]):
            sentiment = "frustrated"
        else:
            sentiment = "neutral"
    else:
        result = llm.invoke(f"Classify sentiment as angry, frustrated, or neutral: {text}").content.lower().strip()
        sentiment = result if result in ["angry", "frustrated", "neutral"] else "neutral"
    return {"sentiment": sentiment, "history": state["history"] + [f"🔍 SENTIMENT: {sentiment.upper()}"]}

# ── NODE 2: Intent Classifier ──
def classifier_node(state: SupportState):
    text = state["ticket_text"].lower()
    if USE_SIMULATION:
        if any(w in text for w in ["refund", "money back", "charge", "charged"]):
            intent = "refund"
        elif any(w in text for w in ["bill", "invoice", "payment", "subscription"]):
            intent = "billing"
        elif any(w in text for w in ["password", "login", "account", "locked"]):
            intent = "account"
        else:
            intent = "technical"
    else:
        result = llm.invoke(f"Classify as refund, billing, account, or technical: {text}").content.lower().strip()
        intent = result if result in ["refund", "billing", "account", "technical"] else "technical"
    return {"intent": intent, "history": state["history"] + [f"🏷️ INTENT: {intent.upper()}"]}

# ── NODE 3: Priority Scorer ──
def priority_node(state: SupportState):
    if state["customer_tier"] == "vip" and state["sentiment"] == "angry":
        priority = "P0_CRITICAL"
    elif state["customer_tier"] == "vip" or state["sentiment"] == "angry":
        priority = "P1_HIGH"
    elif state["sentiment"] == "frustrated":
        priority = "P2_MEDIUM"
    else:
        priority = "P3_LOW"
    return {"priority": priority, "history": state["history"] + [f"⚡ PRIORITY: {priority}"]}

# ── NODE 4-6: Specialized Handlers ──
def refund_handler(state: SupportState):
    return {"response": "💰 Refund of $49.99 initiated. Funds will appear in 3-5 business days. Confirmation: REF-88321.", "history": state["history"] + ["HANDLER: Refund processed → finance queue"]}

def billing_handler(state: SupportState):
    return {"response": "📋 Billing issue logged. Your next invoice has been adjusted. Updated statement will be emailed within 24 hours.", "history": state["history"] + ["HANDLER: Billing adjusted → accounts team"]}

def tech_handler(state: SupportState):
    return {"response": "🔧 Diagnostic packet sent to your device. Engineer assigned: ENG-4421. ETA: 2 hours.", "history": state["history"] + ["HANDLER: Tech support dispatched → engineering"]}

def account_handler(state: SupportState):
    return {"response": "🔑 Account recovery initiated. Temporary access link sent to your registered email. Valid for 30 minutes.", "history": state["history"] + ["HANDLER: Account recovery → security team"]}

# ── NODE 7: VIP Escalation Check ──
def escalation_node(state: SupportState):
    if state["priority"] in ["P0_CRITICAL", "P1_HIGH"]:
        return {"escalated": True, "response": state["response"] + " \n\n🚨 ESCALATED: Manager notified. Direct callback scheduled within 15 minutes.", "history": state["history"] + ["🚨 ESCALATED → Manager queue"]}
    return {"escalated": False, "history": state["history"] + ["✅ Resolved at Tier-1"]}

# ── BUILD THE STATE MACHINE ──
def route_by_intent(state):
    mapping = {"refund": "refund_handler", "billing": "billing_handler", "technical": "tech_handler", "account": "account_handler"}
    return mapping.get(state["intent"], "tech_handler")

builder = StateGraph(SupportState)
builder.add_node("sentiment", sentiment_node)
builder.add_node("classifier", classifier_node)
builder.add_node("priority", priority_node)
builder.add_node("refund_handler", refund_handler)
builder.add_node("billing_handler", billing_handler)
builder.add_node("tech_handler", tech_handler)
builder.add_node("account_handler", account_handler)
builder.add_node("escalation", escalation_node)

builder.set_entry_point("sentiment")
builder.add_edge("sentiment", "classifier")
builder.add_edge("classifier", "priority")
builder.add_conditional_edges("priority", route_by_intent)
builder.add_edge("refund_handler", "escalation")
builder.add_edge("billing_handler", "escalation")
builder.add_edge("tech_handler", "escalation")
builder.add_edge("account_handler", "escalation")
builder.add_edge("escalation", END)
graph = builder.compile()

# ── RUN 3 DIFFERENT TICKETS ──
TICKETS = [
    {"text": "I want my money back NOW! This is unacceptable, I was charged twice for order #8832!", "tier": "vip"},
    {"text": "My app keeps crashing when I try to upload files. Started happening after the latest update.", "tier": "standard"},
    {"text": "I am locked out of my account and I have been waiting for 3 days. Still no help!", "tier": "standard"},
]

all_results = []
for t in TICKETS:
    res = graph.invoke({"ticket_text": t["text"], "customer_tier": t["tier"], "sentiment": "", "intent": "", "priority": "", "history": [], "response": "", "escalated": False})
    all_results.append(res)

In [ ]:
from IPython.display import display, HTML

priority_colors = {"P0_CRITICAL": "#ef4444", "P1_HIGH": "#f97316", "P2_MEDIUM": "#eab308", "P3_LOW": "#22c55e"}
sentiment_icons = {"angry": "😡", "frustrated": "😤", "neutral": "😐"}

tickets_html = ""
for i, r in enumerate(all_results):
    p_color = priority_colors.get(r["priority"], "#666")
    s_icon = sentiment_icons.get(r["sentiment"], "😐")
    esc_badge = '<span style="background:#ef444422; color:#ef4444; padding:2px 8px; border-radius:50px; font-size:9px; margin-left:8px;">ESCALATED</span>' if r["escalated"] else '<span style="background:#22c55e22; color:#22c55e; padding:2px 8px; border-radius:50px; font-size:9px; margin-left:8px;">RESOLVED</span>'
    steps_html = "<br>".join(r["history"])
    resp_html = r["response"].replace("\n", "<br>")

    tickets_html += '<div style="margin-bottom:30px; border:1px solid #1a1e2e; border-radius:20px; overflow:hidden;">'
    # Header
    tickets_html += '<div style="background:#111827; padding:15px 20px; display:flex; align-items:center; justify-content:space-between;">'
    tickets_html += '<div style="font-size:12px; color:#e2e8f0;">TICKET_' + str(i+1) + ' ' + s_icon + esc_badge + '</div>'
    tickets_html += '<div style="font-size:10px; color:' + p_color + '; font-weight:700; background:' + p_color + '22; padding:2px 10px; border-radius:50px;">' + r["priority"] + '</div>'
    tickets_html += '</div>'
    # Body
    tickets_html += '<div style="padding:20px;">'
    tickets_html += '<div style="font-size:14px; color:#cbd5e1; margin-bottom:15px; border-left:3px solid ' + p_color + '; padding-left:12px;">' + r["ticket_text"] + '</div>'
    tickets_html += '<div style="font-size:11px; color:#475569; background:#0a0a0f; padding:12px; border-radius:10px; margin-bottom:15px;">' + steps_html + '</div>'
    tickets_html += '<div style="font-size:13px; color:#e2e8f0; background:#064e3b44; padding:15px; border-radius:12px; border:1px solid #065f46;">' + resp_html + '</div>'
    tickets_html += '</div></div>'

html = '<style>@import url("https://fonts.googleapis.com/css2?family=Inter:wght@400;700&family=Unbounded:wght@800&display=swap");</style>'
html += '<div style="max-width:900px; margin:30px auto; font-family:Inter,sans-serif; background:#0b0b0f; padding:45px; border-radius:40px; border:1px solid #1a1e2e; box-shadow:0 50px 100px rgba(0,0,0,1);">'
html += '<div style="display:flex; align-items:center; justify-content:space-between; margin-bottom:35px;">'
html += '<div style="font-family:Unbounded,sans-serif; font-size:16px; color:#f43f5e; letter-spacing:2px;">LANGGRAPH_SIGNAL_ROUTER</div>'
html += '<div style="font-size:10px; color:#64748b;">7 NODES • 4 INTENTS • CONDITIONAL EDGES</div>'
html += '</div>'
html += tickets_html
html += '</div>'
display(HTML(html))